In [1]:
import sys
sys.path.insert(0, '../../gofher')

import os
import matplotlib.image as mpimg
from astropy.visualization import make_lupton_rgb

from gofher import run_gofher, run_gofher_with_parameters
from visualize import visualize
from file_helper import write_csv,check_if_folder_exists_and_create, construct_csv_dict
from spin_parity import read_spin_parity_galaxies_label_from_csv, standardize_galaxy_name
from sparcfire import read_sparcfire_galaxy_csv, get_ref_band_and_gofher_params

In [2]:
survery_to_use = "sdss" #Note: for sdss we are not using u do to poor quality

BANDS_IN_ORDER = ['g','r','i','z'] #Important: Must stay in order of BLUEST to REDDEST Waveband (Editting this will cause gofher to no longer correctly evaluate redder side of galaxy)
REF_BANDS_IN_ORDER = ['r','i','z','g'] #The prefernce each waveband being choosen as refernce band from highest priority to lowest priority

In [3]:
#figures_to_run_on = ["table2","table3","table4","table5"]
figures_to_run_on = ["figure9"]

In [4]:
#panstarrs:
bin_size = None #None or a positive integer

#sdss:
#NOTE: sdss needs to flip color image
##bin_size = 4 #should be 4
#Source: "The median seeing of all SDSS imaging data (using the psfWidth metric) is 1.32 arcseconds in the r-band."
#"The pixel size in the Sloan Digital Sky Survey (SDSS) is 0.396 arcseconds per pixel" - https://classic.sdss.org/dr3/instruments/imager/
bin_prior_to_param_fitting = True

In [5]:
generate_verbose_csv = True
generate_ebm_csv = False
generate_params_csv = True
generate_visualization = True
save_visualization = True
generate_gamma_csv = True

In [6]:
#Important: Make sure you update these values:
blur_sdss_fits_folder = "E:\\grad_school\\research\\spin_parity_blurring\\sdss_output"
blur_sdss_folder = "E:\\grad_school\\research\\spin_parity_blurring\\sparcfire_sdss_output"
blur_sparcfire_folder = "E:\\grad_school\\research\\spin_parity_blurring\\sparcfire_sdss_output"
path_to_output = "C:\\Users\\school\\Desktop\\github\\gofher-data\\sdss\\blur_folder_9"
gofher_labels_folder = "C:\\Users\\school\\Desktop\\github\\gofher-data\\sdss\\sparcfire_0_25"

In [7]:
def get_blur_folder(the_sn,the_psf):
    return "psf_{}_background_{}".format(str(the_psf),str(the_sn))

def get_sparcfire_galaxy_csv_path(table_name,the_sn,the_psf):
    return os.path.join(blur_sdss_folder,get_blur_folder(the_sn,the_psf),table_name,"G.out","galaxy.csv")

In [8]:
def construct_image(gal):
    for band in ["g","r","i"]:
        if band not in gal.bands:
            the_keys = list(gal.bands.keys())
            return gal[the_keys[0]].data

    g = gal.bands["g"].data
    r = gal.bands["r"].data*0.8
    i = gal.bands["i"].data*0.7

    return make_lupton_rgb(i, r, g, Q=10, stretch=0.3, minimum=0.0)

In [9]:

folder_map = {"table2":"figure8",
              "table3":"figure9",
              "table4":"figure10",
              "table5":"figure11",
              "figure9":"table3"}

def get_fits_path(name,band,blur_folder,figure_to_run_on):
    """the file path of where existing fits files can be found"""
    return os.path.join(blur_sdss_fits_folder,blur_folder,figure_to_run_on,name,"{}_{}.fits".format(name,band))

def get_galaxy_list(blur_folder,figure_to_run_on):
    the_directory = os.path.join(blur_sdss_fits_folder,blur_folder,figure_to_run_on)
    return os.listdir(the_directory)

def get_sparcfire_path(blur_folder,figure_to_run_on): #temp
    return os.path.join(blur_sparcfire_folder,blur_folder,figure_to_run_on,"G.out","galaxy.csv")

def get_path_to_output(blur_folder, figure_to_run_on):
    if figure_to_run_on in folder_map:
        figure_to_run_on = folder_map[figure_to_run_on]
    return os.path.join(path_to_output,blur_folder)

def get_gofher_labels(figure_to_run_on):
    if figure_to_run_on in folder_map:
        figure_to_run_on = folder_map[figure_to_run_on]
    path = os.path.join(gofher_labels_folder,f"{figure_to_run_on}_verbose.csv")
    return construct_csv_dict(path,"name","GOFHER_label")

In [10]:
def get_visulization_save_path_folder(name,figure_to_run_on, blur_folder):
    if figure_to_run_on in folder_map:
        figure_to_run_on = folder_map[figure_to_run_on]
    return os.path.join(path_to_output,blur_folder,figure_to_run_on)

def _ensure_path_exists(path_to_output,make_ouput_folder_if_not_exists=True):
    if not make_ouput_folder_if_not_exists:
        raise ValueError("The path output is not found {} - make sure you update path_to_output".format(path_to_output))
    
    if not os.path.exists(path_to_output):
        os.makedirs(path_to_output)

    

In [11]:
def run_gofher_on_catalog(figure_to_run_on,blur_folder,bulge_disk_f=1.0):
    #paper_labels = get_paper_dark_side_labels(figure_to_run_on)
    gofher_labels = get_gofher_labels(figure_to_run_on)
    sparcfire_gals = read_sparcfire_galaxy_csv(get_sparcfire_path(blur_folder,figure_to_run_on))

    verbose_header = []
    verbose_rows = []

    params_header = []
    params_rows = []

    gamma_header = []
    gamma_rows = []

    i = 1

    path_to_output = get_path_to_output(blur_folder, figure_to_run_on)
    #print(path_to_output)
    #return
    _ensure_path_exists(path_to_output)

    def get_fits_path_cat(name,band):
        """the file path of where existing fits files can be found"""
        return get_fits_path(name,band,blur_folder,figure_to_run_on)

    galaxies = get_galaxy_list(blur_folder,figure_to_run_on)
    for name in galaxies:

        if standardize_galaxy_name(name) not in gofher_labels:
            print("skipping",name)
            continue

        print(name, i,"of",len(galaxies))

        try:
            paper_label = gofher_labels[standardize_galaxy_name(name)]

            ref_band, inital_gofher_params = get_ref_band_and_gofher_params(sparcfire_gals[name],REF_BANDS_IN_ORDER,bulge_disk_f)
            gal = run_gofher_with_parameters(name,get_fits_path_cat,BANDS_IN_ORDER,ref_band,inital_gofher_params,paper_label=paper_label)

            if generate_verbose_csv:
                (header,row) = gal.get_verbose_csv_header_and_row(BANDS_IN_ORDER,paper_label)
                if len(verbose_header) == 0: verbose_header = header
                verbose_rows.append(row)

            if generate_params_csv:
                (header,row) = gal.get_params_csv_header_and_row()
                if len(params_header) == 0: params_header = header
                params_rows.append(row)

            if generate_gamma_csv:
                (header,row) = gal.get_gamma_csv_header_and_row(BANDS_IN_ORDER)
                if len(gamma_header) == 0: gamma_header = header
                gamma_rows.append(row)

            if generate_visualization:
                save_path = ''
                
                if save_visualization:
                    sub_folder = get_visulization_save_path_folder(name,figure_to_run_on, blur_folder)
                    check_if_folder_exists_and_create(sub_folder)
                    save_path = os.path.join(sub_folder,"{}.png".format(name))

                color_image = construct_image(gal)
                visual_string = blur_folder
                visualize(gal,color_image,BANDS_IN_ORDER,paper_label,save_path=save_path,color_flip=(survery_to_use=="sdss"),show_stats=False,visual_string=visual_string)
        except Exception as e:
            print(e)
        i += 1
               
    if generate_verbose_csv:
        verbose_csv_path = os.path.join(path_to_output,"{}_verbose.csv".format(figure_to_run_on))
        write_csv(verbose_csv_path,verbose_header,verbose_rows)

    if generate_params_csv:
        params_csv_path = os.path.join(path_to_output,"{}_params.csv".format(figure_to_run_on))
        write_csv(params_csv_path,params_header,params_rows)

    if generate_gamma_csv:
        params_csv_path = os.path.join(path_to_output,"{}_gamma.csv".format(figure_to_run_on))
        write_csv(params_csv_path,gamma_header,gamma_rows)

In [12]:
#if not os.path.exists(path_to_catalog_data):
#    raise ValueError("The path to the catalog is not found {} - make sure you update path_to_catalog_data".format(path_to_catalog_data))
#sn_list = [0.5, 0.25,0.125,0.0625, 8, 16, 32, 64, 128, 256]
#sn_list = [1,2,4]
sn_list = [0.5, 0.25,0.125,0.0625, 1,2,4,8, 16, 32, 64, 128, 256]
#psf_list = [16.0, 22.6, 32.0, 45.2, 64.0, 90.5, 128.0] #need to do + all figure9/table3 seperate
#psf_list = [4.0, 5.6, 8.0, 11.3]
psf_list = [4.0, 5.6, 8.0, 11.3,16.0, 22.6, 32.0, 45.2, 64.0, 90.5, 128.0]

for sn in sn_list:
    for psf in psf_list:
        for figure_to_run_on in figures_to_run_on:
            blur_folder = get_blur_folder(sn,psf)
            run_gofher_on_catalog(figure_to_run_on,blur_folder,bulge_disk_f=0.25)
            #break
        #break
    #break

IC4566 1 of 14
'IC4566'
NGC2553 2 of 14
'NGC2553'
NGC3344 3 of 14
construct_band: fits_path E:\grad_school\research\spin_parity_blurring\sdss_output\psf_4.0_background_0.5\figure9\NGC3344\NGC3344_z.fits does not exist
NGC3346 4 of 14
NGC3351 5 of 14
NGC3359 6 of 14
zero-size array to reduction operation minimum which has no identity
NGC3367 7 of 14
NGC3381 8 of 14
NGC3395 9 of 14
'NGC3395'
NGC3423 10 of 14
NGC3445 11 of 14
PGC39728 12 of 14
PGC46767 13 of 14
construct_band: fits_path E:\grad_school\research\spin_parity_blurring\sdss_output\psf_4.0_background_0.5\figure9\PGC46767\PGC46767_g.fits does not exist
construct_band: fits_path E:\grad_school\research\spin_parity_blurring\sdss_output\psf_4.0_background_0.5\figure9\PGC46767\PGC46767_z.fits does not exist
PGC49906 14 of 14
IC4566 1 of 14
'IC4566'
NGC2553 2 of 14
'NGC2553'
NGC3344 3 of 14
construct_band: fits_path E:\grad_school\research\spin_parity_blurring\sdss_output\psf_5.6_background_0.5\figure9\NGC3344\NGC3344_z.fits does not

c:\Users\school\Desktop\github\gofher\examples\paper_2\../../gofher\stats.py:23: RuntimeWarning: divide by zero encountered in scalar divide
  initial_alpha = (sample_mean**2) / sample_variance
c:\Users\school\Desktop\github\gofher\gofher_env\Lib\site-packages\scipy\stats\_continuous_distns.py:3133: RuntimeWarning: divide by zero encountered in scalar divide
  aest = (3-s + np.sqrt((s-3)**2 + 24*s)) / (12*s)
c:\Users\school\Desktop\github\gofher\gofher_env\Lib\site-packages\scipy\stats\_continuous_distns.py:3132: RuntimeWarning: invalid value encountered in scalar subtract
  func = lambda a: np.log(a) - sc.digamma(a) - s


IC4566 1 of 14
'IC4566'
NGC2553 2 of 14
'NGC2553'
NGC3344 3 of 14
construct_band: fits_path E:\grad_school\research\spin_parity_blurring\sdss_output\psf_4.0_background_0.25\figure9\NGC3344\NGC3344_z.fits does not exist
NGC3346 4 of 14
NGC3351 5 of 14
NGC3359 6 of 14
zero-size array to reduction operation minimum which has no identity
NGC3367 7 of 14
NGC3381 8 of 14
NGC3395 9 of 14
'NGC3395'
NGC3423 10 of 14
NGC3445 11 of 14
PGC39728 12 of 14
PGC46767 13 of 14
construct_band: fits_path E:\grad_school\research\spin_parity_blurring\sdss_output\psf_4.0_background_0.25\figure9\PGC46767\PGC46767_g.fits does not exist
construct_band: fits_path E:\grad_school\research\spin_parity_blurring\sdss_output\psf_4.0_background_0.25\figure9\PGC46767\PGC46767_z.fits does not exist
PGC49906 14 of 14
IC4566 1 of 14
'IC4566'
NGC2553 2 of 14
'NGC2553'
NGC3344 3 of 14
construct_band: fits_path E:\grad_school\research\spin_parity_blurring\sdss_output\psf_5.6_background_0.25\figure9\NGC3344\NGC3344_z.fits does

c:\Users\school\Desktop\github\gofher\examples\paper_2\../../gofher\visualize.py:138: RuntimeWarning: invalid value encountered in scalar power
  std = ((sum_of_squares-(sum_of/n))/n)**0.5


arange: cannot compute length
PGC49906 14 of 14
zero-size array to reduction operation minimum which has no identity
IC4566 1 of 14
'IC4566'
NGC2553 2 of 14
'NGC2553'
NGC3344 3 of 14
construct_band: fits_path E:\grad_school\research\spin_parity_blurring\sdss_output\psf_90.5_background_32\figure9\NGC3344\NGC3344_z.fits does not exist
NGC3346 4 of 14
NGC3351 5 of 14
zero-size array to reduction operation minimum which has no identity
NGC3359 6 of 14
NGC3367 7 of 14
NGC3381 8 of 14
NGC3395 9 of 14
'NGC3395'
NGC3423 10 of 14
NGC3445 11 of 14
zero-size array to reduction operation minimum which has no identity
PGC39728 12 of 14
zero-size array to reduction operation minimum which has no identity
PGC46767 13 of 14
construct_band: fits_path E:\grad_school\research\spin_parity_blurring\sdss_output\psf_90.5_background_32\figure9\PGC46767\PGC46767_g.fits does not exist
construct_band: fits_path E:\grad_school\research\spin_parity_blurring\sdss_output\psf_90.5_background_32\figure9\PGC46767\PGC46

...

c:\Users\school\Desktop\github\gofher\examples\paper_2\../../gofher\visualize.py:138: RuntimeWarning: invalid value encountered in scalar power
  std = ((sum_of_squares-(sum_of/n))/n)**0.5


IC4566 1 of 14
'IC4566'
NGC2553 2 of 14
'NGC2553'
NGC3344 3 of 14
construct_band: fits_path E:\grad_school\research\spin_parity_blurring\sdss_output\psf_4.0_background_256\figure9\NGC3344\NGC3344_z.fits does not exist
NGC3346 4 of 14
NGC3351 5 of 14
NGC3359 6 of 14
zero-size array to reduction operation minimum which has no identity
NGC3367 7 of 14
NGC3381 8 of 14
NGC3395 9 of 14
'NGC3395'
NGC3423 10 of 14
NGC3445 11 of 14
PGC39728 12 of 14
PGC46767 13 of 14
construct_band: fits_path E:\grad_school\research\spin_parity_blurring\sdss_output\psf_4.0_background_256\figure9\PGC46767\PGC46767_g.fits does not exist
construct_band: fits_path E:\grad_school\research\spin_parity_blurring\sdss_output\psf_4.0_background_256\figure9\PGC46767\PGC46767_z.fits does not exist
PGC49906 14 of 14
IC4566 1 of 14
'IC4566'
NGC2553 2 of 14
'NGC2553'
NGC3344 3 of 14
construct_band: fits_path E:\grad_school\research\spin_parity_blurring\sdss_output\psf_5.6_background_256\figure9\NGC3344\NGC3344_z.fits does not

c:\Users\school\Desktop\github\gofher\examples\paper_2\../../gofher\visualize.py:138: RuntimeWarning: invalid value encountered in scalar power
  std = ((sum_of_squares-(sum_of/n))/n)**0.5


NGC3346 4 of 14
NGC3351 5 of 14
zero-size array to reduction operation minimum which has no identity
NGC3359 6 of 14
NGC3367 7 of 14
NGC3381 8 of 14
NGC3395 9 of 14
'NGC3395'
NGC3423 10 of 14
NGC3445 11 of 14
PGC39728 12 of 14
zero-size array to reduction operation minimum which has no identity
PGC46767 13 of 14
construct_band: fits_path E:\grad_school\research\spin_parity_blurring\sdss_output\psf_90.5_background_256\figure9\PGC46767\PGC46767_g.fits does not exist
construct_band: fits_path E:\grad_school\research\spin_parity_blurring\sdss_output\psf_90.5_background_256\figure9\PGC46767\PGC46767_z.fits does not exist
zero-size array to reduction operation minimum which has no identity
PGC49906 14 of 14
zero-size array to reduction operation minimum which has no identity
IC4566 1 of 14
'IC4566'
NGC2553 2 of 14
'NGC2553'
NGC3344 3 of 14
construct_band: fits_path E:\grad_school\research\spin_parity_blurring\sdss_output\psf_128.0_background_256\figure9\NGC3344\NGC3344_z.fits does not exist


c:\Users\school\Desktop\github\gofher\examples\paper_2\../../gofher\visualize.py:138: RuntimeWarning: invalid value encountered in scalar power
  std = ((sum_of_squares-(sum_of/n))/n)**0.5
